# Detecting Covariate Shift

**The problem**: Your model was trained on data from Distribution A. In production, you're seeing data from Distribution B. The model doesn't know the difference—it just makes predictions. But those predictions might be garbage.

**This notebook**: We'll simulate this scenario and use three different methods to detect when our input distribution has shifted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Our drift detection tools
import sys
sys.path.append('..')
from drift_detection import KSTest, PSI, MMD

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

np.random.seed(42)

## The Scenario

Imagine we've deployed a breast cancer screening AI. It was trained on mammograms from Hospital A—an urban academic medical center with newer digital equipment and a diverse patient population.

Now it's being used at Hospital B—a rural community hospital with older equipment and a different demographic mix (older patients on average).

We'll simulate this with two features:
- **Pixel intensity** (affected by scanner calibration)
- **Patient age** (different demographics)

In [ ]:
# Reference distribution: Hospital A (training data)
n_reference = 2000

ref_intensity = np.random.normal(loc=0.50, scale=0.12, size=n_reference)
ref_age = np.random.normal(loc=52, scale=10, size=n_reference)

reference_data = np.column_stack([ref_intensity, ref_age])
reference_df = pd.DataFrame(reference_data, columns=['pixel_intensity', 'age'])
reference_df['source'] = 'Reference (Hospital A)'

print(f"Reference data: {len(reference_df)} samples")
reference_df.describe()

In [ ]:
# Current distribution: Hospital B (production data)
n_current = 1500

# Different scanner -> different pixel intensity distribution
cur_intensity = np.random.normal(loc=0.55, scale=0.14, size=n_current)

# Older patient population
cur_age = np.random.normal(loc=58, scale=12, size=n_current)

current_data = np.column_stack([cur_intensity, cur_age])
current_df = pd.DataFrame(current_data, columns=['pixel_intensity', 'age'])
current_df['source'] = 'Current (Hospital B)'

print(f"Current data: {len(current_df)} samples")
current_df.describe()

## Visualizing the Shift

Let's see what this drift looks like visually.

In [ ]:
combined_df = pd.concat([reference_df, current_df])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=combined_df, x='pixel_intensity', hue='source', 
            fill=True, alpha=0.3, ax=axes[0])
axes[0].set_title('Pixel Intensity Distribution', fontsize=14)
axes[0].set_xlabel('Normalized Pixel Intensity')

sns.kdeplot(data=combined_df, x='age', hue='source', 
            fill=True, alpha=0.3, ax=axes[1])
axes[1].set_title('Patient Age Distribution', fontsize=14)
axes[1].set_xlabel('Age (years)')

plt.tight_layout()
plt.show()

print("Visual inspection: Both features appear shifted. But is it statistically significant?")

## Method 1: Kolmogorov-Smirnov Test

The KS test compares the cumulative distribution functions of two samples.

**Pros**: Simple, interpretable, no hyperparameters  
**Cons**: Only works on one feature at a time

In [ ]:
ks_detector = KSTest(alpha=0.05)

print("KS Test Results")
print("=" * 50)

for feature in ['pixel_intensity', 'age']:
    result = ks_detector.detect(
        reference_df[feature].values,
        current_df[feature].values
    )
    print(f"\n{feature}:")
    print(f"  KS statistic: {result.statistic:.4f}")
    print(f"  p-value: {result.p_value:.2e}")
    print(f"  Drift detected: {'YES' if result.drift_detected else 'NO'}")

## Method 2: Population Stability Index (PSI)

PSI is the industry standard in banking and insurance.

**Interpretation**:  
- PSI < 0.1: No significant shift  
- PSI 0.1 - 0.2: Moderate shift, investigate  
- PSI > 0.2: Significant shift, take action

In [ ]:
psi_calculator = PSI(n_bins=10, binning='quantile')

print("PSI Results")
print("=" * 50)

for feature in ['pixel_intensity', 'age']:
    result = psi_calculator.calculate(
        reference_df[feature].values,
        current_df[feature].values
    )
    print(f"\n{feature}:")
    print(f"  PSI: {result.psi:.4f}")
    print(f"  Status: {result.drift_level.upper()}")

## Method 3: Maximum Mean Discrepancy (MMD)

MMD compares the full joint distribution, not just individual features.

**Pros**: Works on high-dimensional data, catches multivariate shifts  
**Cons**: Slower, less interpretable

In [ ]:
mmd_detector = MMD()

print("MMD Test Results (Joint Distribution)")
print("=" * 50)

result = mmd_detector.detect(
    reference_data,
    current_data,
    permutation_test=True,
    n_permutations=100
)

print(f"\nMMD: {result.mmd:.4f}")
print(f"p-value: {result.p_value:.4f}")
print(f"Drift detected: {'YES' if result.drift_detected else 'NO'}")

## Summary

All three methods detected drift. In a real deployment, this should trigger:

1. **Alert** the ML ops team
2. **Investigate** the root cause
3. **Decide** whether to recalibrate, retrain, or halt

In [ ]:
print("\n" + "=" * 60)
print(" DRIFT DETECTION SUMMARY")
print("=" * 60)
print("\nAll methods detected significant drift.")
print("Recommended action: Investigate root cause before continuing deployment.")